---
title: Research output
description: Annual publication counts, growth, document types, journal and conference shares, team sizes and fractional counts of the School of Electrical and Computer Engineering, Aristotle University of Thessaloniki, in Scopus since 1981.
---

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "sece_stats").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from sece_stats import config, data
from sece_stats import plotting as dp

dp.apply_theme()
PARTIAL = config.PARTIAL_YEAR   # the year of the export, partial
AS_OF = config.EXPORT_DATE      # the date of the Scopus export

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from sece_stats import metrics, periods

pubs = data.load_publications()
pub_aff = data.load_publication_affiliations()

FIRST_YEAR = int(pubs["year"].min())
LAST_FULL = periods.LAST_FULL   # latest full year (PARTIAL - 1)
START = 1995                    # first year of the per-year share and team-size charts (small counts before)
AS_OF_TEXT = f"{AS_OF.day} {AS_OF:%B %Y}"
NUMBER_WORDS = ("zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen "
                "fifteen sixteen seventeen eighteen nineteen twenty").split()


def join_and(items: list[str]) -> str:
    """'a', 'a and b', 'a, b and c'."""
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " and " + items[-1]


# Five-year periods end with the latest full year (sece_stats.periods) and move with every export;
# the partial year has a label of its own. The names say what each period is in the text.
PERIODS = periods.periods(FIRST_YEAR)        # the full-year periods, oldest first
P_RECENT = periods.period_of(LAST_FULL)      # the latest five full years
P_PREV = periods.period_of(LAST_FULL - 5)    # the five years before them
P_LONG = periods.period_of(LAST_FULL - 15)   # fifteen years earlier: full and fractional counts compared
P_BASE = periods.period_of(LAST_FULL - 20)   # twenty years earlier: baseline of team size and SECE share
P_EARLY = next(p for p in PERIODS if periods.bounds(p)[0] >= START)   # the early output: first period of the charts
# the SECE share in single-institution publications is compared over the periods from 1990 on (few before)
P_SI_FROM = next(p for p in PERIODS if periods.bounds(p)[0] >= 1990)

# Derived columns ------------------------------------------------------------------
type_counts = pubs["doc_type"].value_counts()
MAIN_TYPES = type_counts.index[:4].tolist()   # the four largest Scopus document types
TYPE_ORDER = MAIN_TYPES + ["Other"]
pubs["type_group"] = pubs["doc_type"].where(pubs["doc_type"].isin(MAIN_TYPES), "Other")
pubs["period"] = pubs["year"].map(lambda y: periods.period_of(y, first_year=FIRST_YEAR))
pubs["sece_share"] = metrics.safe_share(pubs["n_sece_authors"], pubs["n_authors"])
full = pubs[pubs["year"] < PARTIAL]           # full years only: trends, shares, averages

SK = dp.CATEGORY_COLORS["source_kind"]
TYPE_COLORS = dp.color_map(
    TYPE_ORDER,
    fixed={"Article": SK["journal"], "Conference paper": SK["conference"], "Book chapter": SK["book"]},
)
other_types = sorted(type_counts.iloc[4:].items(), key=lambda kv: (-kv[1], kv[0]))
other_types_text = join_and([f"{t} ({n})" for t, n in other_types])
# document types of each source type, as listed in the caption of the source-type chart
KIND_TYPES = {"journal": {"Article", "Review", "Letter", "Editorial", "Note", "Erratum", "Data paper"},
              "conference": {"Conference paper"}, "book": {"Book", "Book chapter"}}

# Annual output ----------------------------------------------------------------------
years = pd.RangeIndex(FIRST_YEAR, PARTIAL + 1, name="year")
per_year = pubs.groupby("year").size().reindex(years, fill_value=0)   # years without records = 0
per_year_full = per_year.loc[:LAST_FULL]
ma5 = per_year_full.rolling(5, min_periods=5).mean().dropna()         # trailing 5-year mean

n_pubs = len(pubs)
n_partial = int(per_year.loc[PARTIAL])
zero_years = [int(y) for y in per_year_full.index[per_year_full.eq(0)]]
ranked = per_year_full.sort_values(ascending=False, kind="mergesort")
best_year, second_year, third_year = (int(y) for y in ranked.index[:3])
best_n, second_n, third_n = (int(v) for v in ranked.iloc[:3])
last_n = int(per_year_full.loc[LAST_FULL])
n_2000 = int(per_year.loc[2000])
max_80s = int(per_year.loc[:1989].max())
max_pre_start = int(per_year.loc[:START - 1].max())
share_since_2010 = pubs["year"].ge(2010).mean()   # all records, the partial year included
ma_last = ma5.loc[LAST_FULL]                      # the mean of P_RECENT

decade_rows = []
for d in range(FIRST_YEAR // 10 * 10, LAST_FULL + 1, 10):
    y0, y1 = max(d, FIRST_YEAR), min(d + 9, LAST_FULL)
    n = int(per_year_full.loc[y0:y1].sum())
    decade_rows.append({"decade": f"{d}s", "years": f"{y0}–{y1}" if y1 > y0 else str(y0),
                        "n_years": y1 - y0 + 1, "papers": n, "per_year": n / (y1 - y0 + 1)})
decades = pd.DataFrame(decade_rows)
decades["change"] = decades["per_year"].pct_change()
dec = decades.set_index("decade")
DEC_PREV, DEC_LAST = decades["decade"].iloc[-2:]   # the decade before and the latest decade

# Growth -----------------------------------------------------------------------------


def trend_growth(series: pd.Series) -> tuple[float, float, float]:
    """Annual growth implied by a least-squares fit of ln(count) on year, with its 95% confidence
    interval (t distribution with n - 2 degrees of freedom on the slope's standard error)."""
    x = series.index.to_numpy(dtype=float)
    coef, cov = np.polyfit(x, np.log(series.to_numpy(dtype=float)), 1, cov=True)
    half = stats.t.ppf(0.975, len(x) - 2) * np.sqrt(cov[0, 0])
    return tuple(float(np.expm1(v)) for v in (coef[0], coef[0] - half, coef[0] + half))


# Growth since three fixed calendar years (an analysis choice: since 2000, 2010 and 2015), each span
# ending with the latest full year, so the spans grow by a year with every export.
GROWTH_STARTS = (2000, 2010, 2015)
growth = pd.DataFrame([
    {"span": f"{s}–{LAST_FULL}", "start_n": int(per_year_full.loc[s]), "end_n": last_n,
     "cagr": metrics.cagr_between(per_year_full, s, LAST_FULL),
     **dict(zip(("trend", "trend_lo", "trend_hi"), trend_growth(per_year_full.loc[s:LAST_FULL])))}
    for s in GROWTH_STARTS
])
growth["trend_ci"] = [f"{dp.fmt_pct(lo)[:-1]}–{dp.fmt_pct(hi)}"
                      for lo, hi in zip(growth["trend_lo"], growth["trend_hi"])]
span = dict(zip(GROWTH_STARTS, growth["span"]))
cagr = dict(zip(GROWTH_STARTS, growth["cagr"]))
trend = dict(zip(GROWTH_STARTS, growth["trend"]))
trend_ci = dict(zip(GROWTH_STARTS, growth["trend_ci"]))        # interval text, 'lo–hi%'
trend_lo = dict(zip(GROWTH_STARTS, growth["trend_lo"]))
trend_hi = dict(zip(GROWTH_STARTS, growth["trend_hi"]))
# the shortest span: its length in words, and how clearly its trend differs from no growth
G_SHORT = GROWTH_STARTS[-1]
n_short_text = NUMBER_WORDS[LAST_FULL - G_SHORT + 1].capitalize()
if trend_lo[G_SHORT] <= 0:
    trend_short_zero, trend_short_link = "includes zero growth", "and"
elif trend_lo[G_SHORT] < 0.01:
    trend_short_zero, trend_short_link = "only just excludes zero growth", "and"
else:   # clearly above zero: a contrast with the wide interval, not one more caveat
    trend_short_zero, trend_short_link = "excludes zero growth", "but still"
trend_short_note = f"an interval that {trend_short_zero}"
trend_short_clause = f"{trend_short_link} {trend_short_zero}"   # follows "the interval is wide"

period_n = full.groupby("period").size().reindex(PERIODS)
period_change = period_n.pct_change()       # vs the previous five-year period
recent_n, prev_n = int(period_n[P_RECENT]), int(period_n[P_PREV])
growth_recent = recent_n / prev_n - 1
# the period of the slowest growth since the baseline, and the periods either side of it
P_SLOW = period_change.loc[P_BASE:].idxmin()
assert P_SLOW != P_RECENT, "text: growth picked up again after its slowest period"
P_BEFORE_SLOW, P_AFTER_SLOW = (PERIODS[PERIODS.index(P_SLOW) + k] for k in (-1, 1))

# Document types and source types -------------------------------------------------------
type_n = pd.crosstab(full["period"], full["type_group"])
type_share = pd.crosstab(full["period"], full["type_group"], normalize="index")
art = type_share["Article"]
confp = type_share["Conference paper"]
is_review = full["doc_type"].eq("Review")
review_n = is_review.groupby(full["period"]).sum()
review_share = is_review.groupby(full["period"]).mean()

KINDS = [k for k in data.SOURCE_KIND_ORDER if pubs["source_kind"].eq(k).any()]
kind_year = pd.crosstab(full["year"], full["source_kind"]).reindex(columns=KINDS, fill_value=0)
kind_year_share = kind_year.div(kind_year.sum(axis=1), axis=0)
jour = pd.crosstab(full["period"], full["source_kind"], normalize="index")["journal"]
P_LOW = jour.loc[P_EARLY:].idxmin()   # the lowest journal share since the early output
jour_year = kind_year_share["journal"]
conf_year = kind_year["conference"]
conf_best_year, conf_best_n = int(conf_year.idxmax()), int(conf_year.max())   # record year of conference papers
book_share = pubs["source_kind"].eq("book").mean()
n_book = int(pubs["source_kind"].eq("book").sum())

# Team size --------------------------------------------------------------------------
team_period = full.groupby("period")["n_authors"].agg(
    mean="mean", median="median",
    six_plus=lambda s: s.ge(6).mean(), single=lambda s: s.eq(1).mean())
team_kind = full.groupby(["period", "source_kind"])["n_authors"].mean().unstack()
team_year = (full[full["year"] >= START].groupby("year")["n_authors"]
             .agg(n="size", mean="mean", median="median",
                  q1=lambda s: s.quantile(0.25), q3=lambda s: s.quantile(0.75)))
mean_base, mean_recent = team_period.loc[P_BASE, "mean"], team_period.loc[P_RECENT, "mean"]

# Full and fractional counts ------------------------------------------------------------
counts_period = full.groupby("period").agg(
    full_n=("eid", "size"), frac_n=("sece_share", "sum"), sece_share=("sece_share", "mean"))
frac_recent, frac_prev, frac_long = (counts_period.loc[p, "frac_n"] for p in (P_RECENT, P_PREV, P_LONG))
frac_growth = frac_recent / frac_prev - 1
full_growth_long = recent_n / period_n[P_LONG] - 1
frac_growth_long = frac_recent / frac_long - 1
is_single = full["collab_type"].eq("single-institution")
single_inst_share = full[is_single].groupby("period")["sece_share"].mean().loc[P_SI_FROM:]
single_inst_weight = is_single.groupby(full["period"]).mean()   # share of single-institution publications

# Notes ------------------------------------------------------------------------------
n_explicit = int(pubs["has_sece_affiliation"].sum())
# SECE strings that name another AUTH unit (Physics, Journalism, the medical school) and no electrical
# or computer engineering department: the same keyword check as on Methodology & data.
OTHER_UNIT = r"physic|journalism|dermatolog|school of medicine|medical school"
ECE_DEPT = (r"electrical|electr\.?\s*,?\s*eng|elec\.?\s*comp\.?\s*eng"
            r"|(?:electronics?|electrical)\s+(?:and|&)\s+comp\w*\.?\s*eng|computer\s+eng|\bECE\b")
sece_occ = pub_aff[pub_aff["is_sece"]].assign(
    other_unit=lambda d: d["affiliation"].str.contains(OTHER_UNIT, case=False)
    & ~d["affiliation"].str.contains(ECE_DEPT, case=False))
other_unit_rec = sece_occ.groupby("eid")["other_unit"].all()   # records with only such strings
n_other_unit = int(other_unit_rec.sum())
n_confirmed = int(sece_occ.loc[~sece_occ["other_unit"], "eid"].nunique())
assert n_confirmed == n_explicit - n_other_unit, "SECE records: explicit = confirmed + other AUTH units"
other_unit_physics = (sece_occ.loc[sece_occ["eid"].isin(other_unit_rec.index[other_unit_rec]), "affiliation"]
                      .str.contains("physic", case=False).any())
in_press = (pubs.loc[pubs["pub_stage"].eq("Article in press"), "year"].value_counts()
            .sort_index(ascending=False))
n_in_press = int(in_press.sum())
in_press_text = join_and([f"{dp.fmt_int(n)} dated {y}" for y, n in in_press.items()])
share_english = pubs["language"].eq("English").mean()
other_languages = pubs.loc[pubs["language"].ne("English"), "language"].value_counts()
other_languages_text = join_and([f"{lang} ({n})" for lang, n in other_languages.items()])
partial_types = pubs.loc[pubs["year"].eq(PARTIAL), "doc_type"].value_counts()

# Short names for the numbers quoted in the text ------------------------------------------
pyr_dec_prev, pyr_dec_last = dec.loc[DEC_PREV, "per_year"], dec.loc[DEC_LAST, "per_year"]
chg_dec_last, n_dec_last = dec.loc[DEC_LAST, "change"], int(dec.loc[DEC_LAST, "papers"])
years_dec_last = dec.loc[DEC_LAST, "years"]
art_early, art_low, art_recent = art[P_EARLY], art[P_LOW], art[P_RECENT]
conf_early, conf_low, conf_recent = confp[P_EARLY], confp[P_LOW], confp[P_RECENT]
jour_early, jour_low, jour_recent = jour[P_EARLY], jour[P_LOW], jour[P_RECENT]
# the COVID-19 pandemic (historical years): conference papers dipped in 2020 and 2021
jour_2020, jour_last = jour_year[2020], jour_year[LAST_FULL]
conf_2019, conf_2020, conf_2021, conf_last = (int(conf_year[y]) for y in (2019, 2020, 2021, LAST_FULL))
rev_recent_n, rev_prev_n = int(review_n[P_RECENT]), int(review_n[P_PREV])
rev_recent = review_share[P_RECENT]
med_base, med_recent = team_period.loc[P_BASE, "median"], team_period.loc[P_RECENT, "median"]
med_last = team_year.loc[LAST_FULL, "median"]
six_base, six_recent = team_period.loc[P_BASE, "six_plus"], team_period.loc[P_RECENT, "six_plus"]
single_base, single_recent = team_period.loc[P_BASE, "single"], team_period.loc[P_RECENT, "single"]
jmean_base, jmean_recent = team_kind.loc[P_BASE, "journal"], team_kind.loc[P_RECENT, "journal"]
cmean_base, cmean_recent = team_kind.loc[P_BASE, "conference"], team_kind.loc[P_RECENT, "conference"]
dshare_base = counts_period.loc[P_BASE, "sece_share"]
dshare_recent = counts_period.loc[P_RECENT, "sece_share"]
si_min, si_max = single_inst_share.min(), single_inst_share.max()
n_partial_art = int(partial_types.get("Article", 0))
n_partial_conf = int(partial_types.get("Conference paper", 0))
zero_years_text = join_and([str(y) for y in zero_years]) if zero_years else ""

# The text describes these patterns of the current snapshot: fail loudly if new data changes them.
assert FIRST_YEAR == 1981, "description: records since 1981"
assert zero_years, "text: some early years have no records"
assert max_80s < n_2000 < best_n, "text: annual output grew from the 1980s to 2000 and to the record"
assert best_year == LAST_FULL, "text: the latest full year is the record year"
assert ma5.idxmax() == LAST_FULL, "text: the five-year average reached its highest in the latest period"
assert decades["change"].iloc[1:].gt(0).all(), "text: mean annual output rose in every decade"
assert dec.loc[DEC_LAST, "n_years"] < 10, "text: the latest decade is not complete yet ('so far')"
assert all(cagr[s] > 0 and trend[s] > 0 for s in GROWTH_STARTS), "text: output grew over every span"
assert all(trend[s] < cagr[s] for s in (2010, 2015)), "text: trend growth below end-point CAGR"
assert (growth["trend_lo"] < growth["trend"]).all() and (growth["trend"] < growth["trend_hi"]).all(), \
    "the interval brackets the trend"
assert trend_hi[G_SHORT] - trend_lo[G_SHORT] > trend[G_SHORT], "text: the shortest span's interval is wide"
assert period_change[P_BEFORE_SLOW] > period_change[P_SLOW] > 0 \
    and period_change[P_AFTER_SLOW] > period_change[P_SLOW], "text: growth rose, slowed, then picked up again"
assert growth_recent > 0 and 0 < frac_growth < growth_recent, "text: fractional counts grew more slowly"
assert 0 < frac_growth_long < full_growth_long, "text: fractional counts grew more slowly over fifteen years"
assert art_early > 0.5, "text: articles dominated the early output"
assert confp.loc[P_EARLY:].idxmax() == P_LOW and conf_low > conf_early, \
    "text: conference share rose to its peak in the period of the lowest journal share"
assert type_n.loc[P_LOW, "Conference paper"] > type_n.loc[P_LOW, "Article"], \
    "text: conference papers outnumbered articles"
assert art_recent > art_low and conf_recent < conf_low and jour_recent > jour_low, \
    "text: journals have since regained ground"
assert rev_recent_n > rev_prev_n and rev_recent < 0.05, "text: reviews are a small type but more common"
assert jour_year.loc[2020:LAST_FULL].idxmax() == 2020 and jour_last < jour_2020, \
    "text: the journal share has eased since 2020"
assert max(conf_2020, conf_2021) < conf_2019 < conf_year.loc[2022:LAST_FULL].min(), \
    "text: conference papers dipped in 2020 and 2021 and recovered"
assert conf_best_year > 2021, "text: conference papers reached their record after the pandemic"
assert all(set(pubs.loc[pubs["source_kind"].eq(k), "doc_type"]) <= KIND_TYPES[k] for k in KINDS), \
    "caption: document types of each source type"
assert book_share < 0.05, "text: books and book chapters remain marginal"
assert team_period.loc[P_EARLY:, "mean"].is_monotonic_increasing, "text: teams have grown steadily"
assert med_recent > med_base and six_recent > six_base and single_recent < single_base, \
    "text: larger teams, fewer single-author publications"
assert jmean_recent > jmean_base and cmean_recent > cmean_base, "text: teams grew in both kinds"
assert dshare_recent < dshare_base and si_max - si_min < dshare_base - dshare_recent, \
    "text: the SECE share fell overall but stayed level in single-institution publications"
assert single_inst_weight[P_RECENT] < single_inst_weight[P_BASE], \
    "text: growing weight of publications with external co-authors"
assert other_unit_physics, "text: the other AUTH units include the Department of Physics"
assert n_partial_conf < conf_last / 2, "text: the partial year's conference papers are mostly not indexed yet"
assert n_in_press > 0, "text: some records are articles in press"

How much does the School publish, how fast has its output grown, in which forms and in teams of
what size? This page counts the {eval}`dp.fmt_int(n_pubs)` Scopus records of the School of
Electrical and Computer Engineering (SECE) of AUTH dated from {eval}`str(FIRST_YEAR)` up to the export
of {eval}`AS_OF_TEXT`. Each record counts once, however many co-authors it has (full counting),
except in the last section, which weights each record by its share of SECE authors.
{eval}`str(PARTIAL)` is a partial year: it is included in totals and marked in the charts of annual
counts, but left out of trends, growth rates, averages and shares by year or period.

In [ ]:
dp.kpi_html([
    dp.KPI("Publications", n_pubs, f"{FIRST_YEAR}–{PARTIAL} ({PARTIAL} to {AS_OF.day} {AS_OF:%B})"),
    dp.KPI(f"Publications in {LAST_FULL}", last_n,
           "latest full year; best on record" if best_year == LAST_FULL else "latest full year"),
    dp.KPI(f"Growth, {P_RECENT} vs {P_PREV}", dp.fmt_pct(growth_recent, signed=True),
           f"{dp.fmt_int(recent_n)} vs {dp.fmt_int(prev_n)} publications"),
    dp.KPI("Mean authors per publication", dp.fmt_num(mean_recent, 1),
           f"{P_RECENT}; {dp.fmt_num(mean_base, 1)} in {P_BASE}"),
], min_width=140)

:::{important} Key findings
- Output reached a record {eval}`dp.fmt_int(best_n)` publications in {eval}`str(best_year)`, and
  {eval}`dp.fmt_pct(share_since_2010)` of all records date from 2010 or later.
- {eval}`P_RECENT` produced {eval}`dp.fmt_int(recent_n)` publications, {eval}`dp.fmt_pct(growth_recent)`
  more than {eval}`P_PREV`. On a log-linear trend, output has grown by {eval}`dp.fmt_pct(trend[2015])` a year
  since 2015 (95% CI {eval}`trend_ci[2015]`, {eval}`trend_short_note`), and by
  {eval}`dp.fmt_pct(cagr[2015])` a year between the end points 2015 and {eval}`str(LAST_FULL)`.
- Conference papers rose from {eval}`dp.fmt_pct(conf_early)` of output in {eval}`P_EARLY` to
  {eval}`dp.fmt_pct(conf_low)` in {eval}`P_LOW`; journals have since regained ground, from
  {eval}`dp.fmt_pct(jour_low)` of output in {eval}`P_LOW` to {eval}`dp.fmt_pct(jour_recent)` in {eval}`P_RECENT`.
  Year by year, though, the journal share has eased from {eval}`dp.fmt_pct(jour_2020)` in 2020, when
  conference papers dipped during the COVID-19 pandemic, to {eval}`dp.fmt_pct(jour_last)` in {eval}`str(LAST_FULL)`.
- Teams have grown: a publication had {eval}`dp.fmt_num(mean_base, 1)` authors on average in {eval}`P_BASE`
  and {eval}`dp.fmt_num(mean_recent, 1)` in {eval}`P_RECENT`, and the share with six or more authors rose from
  {eval}`dp.fmt_pct(six_base)` to {eval}`dp.fmt_pct(six_recent)`.
- Weighted by the share of SECE authors on each publication, output grew by
  {eval}`dp.fmt_pct(frac_growth)` from {eval}`P_PREV` to {eval}`P_RECENT`, against
  {eval}`dp.fmt_pct(growth_recent)` in full counts.
:::

In [ ]:
#| label: cell-output-per-year
annual = per_year.rename("n").reset_index()
ma_all = ma5.reindex(per_year.index)   # NaN for the first four years and the partial year
# The average is shown in the bars' hover and the line is not hoverable: on narrow screens a
# hoverable line would take over the unified hover of the neighbouring partial-year bar.
bar_hover = ["%{y:,.0f}" if pd.isna(v) else f"%{{y:,.0f}} (5-year average {v:,.1f})" for v in ma_all]
fig = dp.year_bar(annual, "year", "n", partial_year=PARTIAL, as_of=AS_OF, value_title="Publications")
fig.update_traces(selector=dict(type="bar"), name="Publications", showlegend=True, hovertemplate=bar_hover)
fig.add_scatter(x=ma5.index, y=ma5.round(1), mode="lines", name="5-year average",
                line=dict(color=dp.CATEGORICAL[1], width=2),
                hoverinfo="skip")   # no hovertemplate: plotly.js ignores hoverinfo when one is set
fig.update_yaxes(rangemode="tozero")   # the line would otherwise pad the axis below zero
fig.show()

## Publications per year

:::{figure} #cell-output-per-year
:label: fig-output-per-year
Publications per year, with the trailing five-year average (mean of the five years ending in each
year, full years only). {eval}`str(PARTIAL)` is partial (records to {eval}`AS_OF_TEXT`).
:::

As [](#fig-output-per-year) shows, annual output grew from at most {eval}`dp.fmt_int(max_80s)`
publications a year in the 1980s to
{eval}`dp.fmt_int(n_2000)` in 2000 and a record {eval}`dp.fmt_int(best_n)` in {eval}`str(best_year)`,
followed by {eval}`str(second_year)` ({eval}`dp.fmt_int(second_n)`) and {eval}`str(third_year)`
({eval}`dp.fmt_int(third_n)`). The five-year average reached {eval}`dp.fmt_num(ma_last, 1)` publications
a year in {eval}`P_RECENT`, and {eval}`dp.fmt_pct(share_since_2010)` of all records date from 2010 or later.
Early counts are small (no records are dated {eval}`zero_years_text`, and no year before
{eval}`str(START)` has more than {eval}`dp.fmt_int(max_pre_start)`) and should be read with care, as
Scopus covers older literature less completely.

In [ ]:
#| label: cell-output-decades
dp.table_html(
    decades,
    columns={"decade": "Decade", "years": "Years", "papers": "Publications",
             "per_year": "Mean per year", "change": "Change in mean per year"},
    formats={"per_year": ",.1f", "change": lambda v: dp.fmt_pct(v, signed=True)},
)

:::{figure} #cell-output-decades
:kind: table
:label: tbl-output-decades
Publications by decade. The {eval}`DEC_LAST` cover the full years {eval}`years_dec_last`; the
{eval}`dp.fmt_int(n_partial)` records dated {eval}`str(PARTIAL)` (partial year) are not included.
:::

Mean annual output has risen in every decade ([](#tbl-output-decades)). So far the {eval}`DEC_LAST`
have averaged {eval}`dp.fmt_num(pyr_dec_last, 1)` publications a year, {eval}`dp.fmt_pct(chg_dec_last)` more
than the {eval}`DEC_PREV` ({eval}`dp.fmt_num(pyr_dec_prev, 1)`), and they already account for
{eval}`dp.fmt_int(n_dec_last)` publications.

In [ ]:
#| label: cell-output-growth
dp.table_html(
    growth,
    columns={"span": "Years", "start_n": "First year", "end_n": str(LAST_FULL), "cagr": "CAGR",
             "trend": "Trend", "trend_ci": "95% CI"},
    formats={"cagr": "pct", "trend": "pct"},
    numeric=["start_n", "end_n", "cagr", "trend", "trend_ci"],
)

## Growth

:::{figure} #cell-output-growth
:kind: table
:label: tbl-output-growth
Annual growth of output. First year and {eval}`str(LAST_FULL)`: publications in the first year of the
span and in {eval}`str(LAST_FULL)`. CAGR: compound annual growth rate between these two end points.
Trend: annual rate implied by a least-squares fit of the logarithm of annual publications on year,
using every year of the span; 95% CI: its 95% confidence interval.
:::

Measured between end points ([](#tbl-output-growth)), output grew by {eval}`dp.fmt_pct(cagr[2000])` a
year from 2000 to {eval}`str(LAST_FULL)`,
{eval}`dp.fmt_pct(cagr[2010])` from 2010 and {eval}`dp.fmt_pct(cagr[2015])` from 2015. End-point rates
depend on the two years chosen: {eval}`str(best_year)` is the record year, so the trend fits, which use
every year, give lower rates for the recent spans ({eval}`dp.fmt_pct(trend[2010])` a year since 2010,
95% CI {eval}`trend_ci[2010]`, and {eval}`dp.fmt_pct(trend[2015])` since 2015, {eval}`trend_ci[2015]`).
{eval}`n_short_text` years are few for a trend: the {eval}`span[2015]` interval is wide
{eval}`trend_short_clause`.

Compared with the previous five years, output rose by {eval}`dp.fmt_pct(period_change[P_BEFORE_SLOW])` in
{eval}`P_BEFORE_SLOW`; growth then slowed to {eval}`dp.fmt_pct(period_change[P_SLOW])` in {eval}`P_SLOW`
and picked up again in {eval}`P_AFTER_SLOW` ({eval}`dp.fmt_pct(period_change[P_AFTER_SLOW], signed=True)`).
The latest five years, {eval}`P_RECENT`, produced {eval}`dp.fmt_int(recent_n)` publications against
{eval}`dp.fmt_int(prev_n)` in {eval}`P_PREV` ({eval}`dp.fmt_pct(growth_recent, signed=True)`).

In [ ]:
#| label: cell-output-types-year
types_year = pubs.groupby(["year", "type_group"]).size().rename("n").reset_index()
types_year["order"] = types_year["type_group"].map({t: i for i, t in enumerate(TYPE_ORDER)})
types_year = types_year.sort_values(["order", "year"]).drop(columns="order")   # legend/stack order
fig = dp.year_bar(types_year, "year", "n", color="type_group", colors=TYPE_COLORS,
                  partial_year=PARTIAL, as_of=AS_OF, value_title="Publications")
fig.show()

## Document types

:::{figure} #cell-output-types-year
:label: fig-output-types-year
Publications per year by Scopus document type. Other: {eval}`other_types_text`.
{eval}`str(PARTIAL)` is partial (records to {eval}`AS_OF_TEXT`).
:::

In [ ]:
#| label: cell-output-types-period
tp = full.groupby(["period", "type_group"]).size().rename("n").reset_index()
tp["share"] = tp["n"] / tp.groupby("period")["n"].transform("sum")
ticks = {p: f"{p}<br>n = {dp.fmt_int(period_n[p])}" for p in PERIODS}
tp["period_label"] = tp["period"].map(ticks)
fig = px.bar(
    tp, x="period_label", y="share", color="type_group", custom_data=["n"],
    color_discrete_map=TYPE_COLORS,
    category_orders={"type_group": TYPE_ORDER, "period_label": [ticks[p] for p in PERIODS]},
    labels={"share": "Share of publications", "period_label": ""},
)
fig.update_traces(hovertemplate="%{y:.1%} (%{customdata[0]:,} publications)")
fig.update_layout(hovermode="x unified", legend_title_text="", margin=dict(t=24))
fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.show()

:::{figure} #cell-output-types-period
:label: fig-output-types-period
Document-type shares by five-year period (n = publications in the period). {eval}`str(PARTIAL)` is not
shown: the year is partial and its conference papers are mostly not indexed yet.
:::

[](#fig-output-types-year) shows the yearly counts by document type and [](#fig-output-types-period)
the shares by five-year period. Journal articles dominated the early output
({eval}`dp.fmt_pct(art_early)` of publications in {eval}`P_EARLY`). Conference papers then grew quickly,
from {eval}`dp.fmt_pct(conf_early)` in {eval}`P_EARLY` to {eval}`dp.fmt_pct(conf_low)` in {eval}`P_LOW`,
when they outnumbered articles. Articles have since regained ground
({eval}`dp.fmt_pct(art_recent)` of publications in {eval}`P_RECENT`, conference papers
{eval}`dp.fmt_pct(conf_recent)`). Reviews remain a small type but have become more common:
{eval}`dp.fmt_int(rev_recent_n)` in {eval}`P_RECENT` ({eval}`dp.fmt_pct(rev_recent)`), against
{eval}`dp.fmt_int(rev_prev_n)` in {eval}`P_PREV`.

In [ ]:
#| label: cell-output-kind-share
KIND_LABEL = {k: k.capitalize() for k in KINDS}                  # "Journal", "Conference", "Book"
ks = kind_year.loc[START:].stack().rename("n").reset_index()   # year, source_kind, n
ks["share"] = ks["n"] / ks.groupby("year")["n"].transform("sum")
ks["source_kind"] = ks["source_kind"].map(KIND_LABEL)
fig = dp.year_line(ks, "year", "share", color="source_kind",
                   colors={KIND_LABEL[k]: SK[k] for k in KINDS},
                   value_title="Share of publications", value_format=".1%")
for tr in fig.data:   # add the count to the hover
    counts = ks[ks["source_kind"] == tr.name].set_index("year")["n"].reindex(list(tr.x))
    tr.customdata = counts.to_numpy().reshape(-1, 1)
    tr.hovertemplate = "%{y:.1%} (%{customdata[0]:,})"
fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.show()

## Journals, conferences and books

:::{figure} #cell-output-kind-share
:label: fig-output-kind-share
Share of publications by source type, {eval}`str(START)`–{eval}`str(LAST_FULL)` (number of
publications in the hover). The source type follows the Scopus document type: journal = articles, reviews,
letters, editorials, notes, errata and data papers; conference = conference papers; book = books and
book chapters. Earlier years have at most {eval}`dp.fmt_int(max_pre_start)` publications each;
{eval}`str(PARTIAL)` is not shown (partial year, conference papers mostly not indexed yet).
:::

Is output shifting toward journals? Compared with {eval}`P_LOW`, yes ([](#fig-output-kind-share)). The
journal share fell from {eval}`dp.fmt_pct(jour_early)` in {eval}`P_EARLY` to a low of
{eval}`dp.fmt_pct(jour_low)` in {eval}`P_LOW` and recovered to {eval}`dp.fmt_pct(jour_recent)` in
{eval}`P_RECENT`. Year by year, however, it has eased from {eval}`dp.fmt_pct(jour_2020)` in 2020 to
{eval}`dp.fmt_pct(jour_last)` in {eval}`str(LAST_FULL)` as conference output recovered: conference papers
fell from {eval}`dp.fmt_int(conf_2019)` in 2019 to {eval}`dp.fmt_int(conf_2020)` in 2020 and
{eval}`dp.fmt_int(conf_2021)` in 2021, coinciding with the COVID-19 pandemic, when many conferences were
cancelled or moved online, and reached a record {eval}`dp.fmt_int(conf_best_n)` in
{eval}`str(conf_best_year)`. Books and book chapters remain marginal ({eval}`dp.fmt_int(n_book)` records,
{eval}`dp.fmt_pct(book_share)` of all).

In [ ]:
#| label: cell-output-team-size
ty = team_year
band = "rgba(57, 135, 229, 0.18)"   # dp.CATEGORICAL[0], translucent
fig = go.Figure()
fig.add_scatter(x=ty.index, y=ty["q1"], mode="lines", line=dict(width=0), hoverinfo="skip",
                showlegend=False, legendgroup="iqr")
fig.add_scatter(x=ty.index, y=ty["q3"], mode="lines", line=dict(width=0), fill="tonexty",
                fillcolor=band, name="Middle 50% of publications", legendgroup="iqr",
                customdata=ty[["q1", "q3"]].to_numpy(),
                hovertemplate="%{customdata[0]:.2~f}–%{customdata[1]:.2~f}")
fig.add_scatter(x=ty.index, y=ty["median"], mode="lines", name="Median",
                line=dict(color=dp.CATEGORICAL[0], width=2), hovertemplate="%{y:.1~f}")
fig.add_scatter(x=ty.index, y=ty["mean"], mode="lines", name="Mean",
                line=dict(color=dp.CATEGORICAL[1], width=2), hovertemplate="%{y:.2f}")
fig.update_layout(hovermode="x unified", margin=dict(t=24))
fig.update_yaxes(title_text="Authors per publication", rangemode="tozero")
dp.style_year_axis(fig)
fig.show()

## Team size

:::{figure} #cell-output-team-size
:label: fig-output-team-size
Authors per publication by year, {eval}`str(START)`–{eval}`str(LAST_FULL)`: median, mean and
interquartile range (the middle 50% of publications). {eval}`str(PARTIAL)` is not shown (partial year).
:::

Teams have grown steadily ([](#fig-output-team-size)). The mean number of authors per publication
rose from {eval}`dp.fmt_num(mean_base, 1)` in {eval}`P_BASE` to {eval}`dp.fmt_num(mean_recent, 1)` in
{eval}`P_RECENT`, and the median from {eval}`dp.fmt_num(med_base, 0)` to {eval}`dp.fmt_num(med_recent, 0)`
(in {eval}`str(LAST_FULL)`: {eval}`dp.fmt_num(med_last, 0)`). The share of publications with six or more
authors went from {eval}`dp.fmt_pct(six_base)` to {eval}`dp.fmt_pct(six_recent)`, while single-author
publications fell from {eval}`dp.fmt_pct(single_base)` to {eval}`dp.fmt_pct(single_recent)`. The growth
is not a side effect of the changing document mix: the mean team size rose both in journal publications
({eval}`dp.fmt_num(jmean_base, 1)` to {eval}`dp.fmt_num(jmean_recent, 1)`) and in conference papers
({eval}`dp.fmt_num(cmean_base, 1)` to {eval}`dp.fmt_num(cmean_recent, 1)`).

In [ ]:
#| label: cell-output-fractional
COUNT_LABELS = {"full_n": "Full count", "frac_n": "Fractional count (SECE share of authors)"}
fy = (pubs[pubs["year"] >= START].groupby("year")
      .agg(full_n=("eid", "size"), frac_n=("sece_share", "sum"))
      .rename(columns=COUNT_LABELS))
fl = fy.stack().rename("value").rename_axis(["year", "count"]).reset_index()
fig = dp.year_line(
    fl, "year", "value", color="count",
    colors={COUNT_LABELS["full_n"]: dp.INK["muted"], COUNT_LABELS["frac_n"]: dp.CATEGORICAL[0]},
    partial_year=PARTIAL, as_of=AS_OF, value_title="Publications", value_format=",.0f",
)
fig.update_yaxes(rangemode="tozero")
fig.show()

## Full and fractional counts

:::{figure} #cell-output-fractional
:label: fig-output-fractional
Publications per year, {eval}`str(START)`–{eval}`str(PARTIAL)`, counted in full (each publication = 1)
and fractionally (each publication = the share of its authors with an SECE affiliation).
{eval}`str(PARTIAL)` is partial (records to {eval}`AS_OF_TEXT`).
:::

Full counting credits the School with a whole publication however many outside co-authors it has.
Weighting each publication by the share of its authors who give an SECE affiliation yields
{eval}`dp.fmt_int(frac_recent)` publication equivalents in {eval}`P_RECENT`, {eval}`dp.fmt_pct(frac_growth)`
more than in {eval}`P_PREV` ({eval}`dp.fmt_int(frac_prev)`), against {eval}`dp.fmt_pct(growth_recent)` in
full counts ([](#fig-output-fractional)). Between {eval}`P_LONG` and {eval}`P_RECENT`, full counts rose by
{eval}`dp.fmt_pct(full_growth_long)` and fractional counts by {eval}`dp.fmt_pct(frac_growth_long)`. The
average SECE share of a publication's authors fell from {eval}`dp.fmt_pct(dshare_base)` in {eval}`P_BASE`
to {eval}`dp.fmt_pct(dshare_recent)` in {eval}`P_RECENT`. In single-institution publications it stayed
between {eval}`dp.fmt_pct(si_min)` and {eval}`dp.fmt_pct(si_max)` in every five-year period from
{eval}`P_SI_FROM` on, so the decline reflects the growing weight of publications with external
co-authors (see [Collaboration](collaboration.ipynb)) rather than a drift in how SECE authors are
identified.

## Notes & caveats

- **Coverage.** Counts are the records kept from the results of the Scopus affiliation query (see
  [Methodology & data](methodology.ipynb)). {eval}`dp.fmt_int(n_explicit)` of the
  {eval}`dp.fmt_int(n_pubs)` records carry an SECE affiliation string; in {eval}`dp.fmt_int(n_other_unit)` of
  them it names another AUTH unit (for example the Department of Physics), so
  {eval}`dp.fmt_int(n_confirmed)` are confirmed School records. All are counted. Work that Scopus does
  not index (many national conferences, theses, patents, technical reports) is not counted.
- **Partial year.** The export is from {eval}`AS_OF_TEXT`. Of the {eval}`dp.fmt_int(n_partial)` records
  dated {eval}`str(PARTIAL)`, {eval}`dp.fmt_int(n_partial_art)` are articles and only
  {eval}`dp.fmt_int(n_partial_conf)` are conference papers (against {eval}`dp.fmt_int(conf_last)` in
  {eval}`str(LAST_FULL)`): conference proceedings usually reach Scopus months after the event.
  {eval}`str(PARTIAL)` is therefore counted in the totals and in shares of all records, but left out of
  trends, growth rates, averages and shares by year or period.
- **Articles in press.** {eval}`dp.fmt_int(n_in_press)` records are articles in press
  ({eval}`in_press_text`); their final publication year may still change.
- **Language.** {eval}`dp.fmt_pct(share_english)` of the records are in English; the others are in
  {eval}`other_languages_text`.
- **Document types** are those assigned by Scopus. The source type (journal, conference, book) is
  derived from the document type, not from the venue, so a conference paper published in a journal
  issue counts as conference output.
- **Authors** are the author positions listed in Scopus. SECE authors are identified from their
  affiliation text, so the fractional counts depend on how authors report their affiliation (see
  [Research community](people.ipynb)).
- **Growth rates** are descriptive and sensitive to the years compared: a CAGR uses only its two end
  years, a trend fit uses every year of the span.